# 🚀 MedSeg-XAI: Master Integration & Clinical Prototype Notebook
**Unified Platform**: Connecting all four modules agreed upon by the team:
1. `load_case(case_id)` -> Returns image, true_mask, box (Pranav)
2. `segment(image, box)` -> Returns predicted binary mask (Ayush)
3. `explain(image, box)` -> Returns attention heatmap (Explainability Lead)
4. `audit(image, box)` -> Returns MPRT similarity scores & safety verdict (Niyati)

Generates the final Cohort Results Table (Dice per case) and launches the interactive Gradio demo with a public shareable URL.

In [ ]:
# Cell 1: Mount Google Drive & Add Code Directory to Python Path
from google.colab import drive
import sys
import os

drive.mount('/content/drive')

CODE_DIR = '/content/drive/MyDrive/medsegxai/code'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print('Python Path Configured. Importing team modules...')
from load_case import load_case
from segment import segment, compute_dice
from explain import explain
from audit import audit
print('SUCCESS: All 4 team modules loaded cleanly!')

In [ ]:
# Cell 2: Install UI & Visualization Dependencies
!pip install -q gradio pandas matplotlib scipy

In [ ]:
# Cell 3: Batch Pipeline Evaluation Across All 5 Cases
import pandas as pd
import numpy as np

results = []
for i in range(1, 6):
    case_id = f'case_{i:02d}'
    img, true_mask, box = load_case(case_id)
    pred_mask = segment(img, box)
    dice = compute_dice(pred_mask, true_mask)
    heatmap = explain(img, box)
    scores, verdict = audit(img, box)
    
    results.append({
        'Case ID': f'BraTS_Case_{i:02d}',
        'Slice Dim': f'{img.shape[0]}x{img.shape[1]}',
        'Tumor Pixels': int(np.sum(true_mask > 0)),
        'Dice Score': round(dice, 4),
        'Clean SSIM': 1.000,
        'Randomized SSIM': round(scores['Stage 3 (Full Cascading Randomization)'], 4),
        'MPRT Safety Verdict': verdict
    })

df = pd.DataFrame(results)
print('=== COHORT BENCHMARK RESULTS TABLE ===')
display(df)

# Save table to outputs/
df.to_csv('/content/drive/MyDrive/medsegxai/outputs/cohort_benchmark_results.csv', index=False)
print('Saved results to outputs/cohort_benchmark_results.csv')

In [ ]:
# Cell 4: Launch Interactive Gradio Demo (Shareable Link)
demo_script = '/content/drive/MyDrive/medsegxai/demo.py'
if os.path.exists(demo_script):
    from demo import build_app
    app = build_app()
    app.launch(share=True, debug=True)
else:
    print('demo.py not found at:', demo_script)